In [5]:
from datetime import datetime, timedelta, timezone

import pandas as pd
from trino.auth import OAuth2Authentication
from trino.dbapi import connect

from data_engineering.utils import OUTPUT_DIR


conn = connect(
    host="trino.opensky-network.org",
    port=443,
    user="goodejonah",
    catalog="minio",
    schema="osky",
    http_scheme="https",
    auth=OAuth2Authentication(),
    timezone="UTC",
)

date = datetime(2024, 6, 1, tzinfo=timezone.utc)

base_output_dir = (
    OUTPUT_DIR
    / "data"
    / "raw"
    / "opensky"
    / "trino-downloads"
    / "state_vectors_data4"
)

for hour in range(24):
    start = date + timedelta(hours=hour)

    output_dir = (
        base_output_dir
        / f"year={start.year}"
        / f"month={start.month:02d}"
        / f"day={start.day:02d}"
        / f"hour={start.hour:02d}"
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "data.parquet"

    if output_path.exists():
        print(f"Skipping {start}: already downloaded")
        continue

    # state_vectors_data4 is physically partitioned by this Unix-hour value.
    hour_timestamp = int(start.timestamp())

    print(f"Downloading {start}")

    cursor = conn.cursor()
    cursor.execute(
        f"""
        SELECT *
        FROM state_vectors_data4
        WHERE hour = {hour_timestamp}
        """
    )

    rows = cursor.fetchall()
    columns = [column[0] for column in cursor.description]

    df = pd.DataFrame(rows, columns=columns)

    df.to_parquet(
        output_path,
        index=False,
    )

    print(f"Saved {len(df):,} rows -> {output_path}")

Open the following URL in browser for the external authentication:
https://trino.opensky-network.org/oauth2/token/initiate/fd40b79359a9f652d053580535e1d7c2f58ef53b189cb94a44f5b5562ac8dd0a


KeyboardInterrupt: 

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from trino.auth import OAuth2Authentication
from trino.dbapi import connect

from data_engineering.utils import OUTPUT_DIR


BATCH_SIZE = 100_000

conn = connect(
    host="trino.opensky-network.org",
    port=443,
    user="goodejonah",
    catalog="minio",
    schema="osky",
    http_scheme="https",
    auth=OAuth2Authentication(),
    timezone="UTC",

    # Use compressed spooling if supported by the OpenSky server.
    encoding=["json+zstd", "json"],
)

date = datetime(2024, 6, 1, tzinfo=timezone.utc)

base_output_dir = (
    OUTPUT_DIR
    / "data"
    / "raw"
    / "opensky"
    / "trino-downloads"
    / "state_vectors_data4"
)


def rows_to_arrow(
    rows: list[tuple],
    columns: list[str],
) -> pa.Table:
    """
    Convert one Trino batch into an Arrow table.

    Trino's DB-API client already returns Python tuples, so this is the
    earliest point at which we can move the data into Arrow.
    """
    column_values = zip(*rows)

    return pa.Table.from_arrays(
        [pa.array(values) for values in column_values],
        names=columns,
    )


for hour in range(24):
    start = date + timedelta(hours=hour)

    output_dir = (
        base_output_dir
        / f"year={start.year}"
        / f"month={start.month:02d}"
        / f"day={start.day:02d}"
        / f"hour={start.hour:02d}"
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "data.parquet"
    temp_path = output_dir / "data.parquet.tmp"

    if output_path.exists():
        print(f"Skipping {start}: already downloaded")
        continue

    # Remove an incomplete file from a previous failed download.
    temp_path.unlink(missing_ok=True)

    hour_timestamp = int(start.timestamp())

    print(f"Downloading {start}")

    cursor = conn.cursor()

    try:
        cursor.execute(
            f"""
            SELECT *
            FROM state_vectors_data4
            WHERE hour = {hour_timestamp}
            """
        )

        columns = [column[0] for column in cursor.description]

        writer: pq.ParquetWriter | None = None
        total_rows = 0

        try:
            while True:
                rows = cursor.fetchmany(BATCH_SIZE)

                if not rows:
                    break

                table = rows_to_arrow(rows, columns)

                if writer is None:
                    writer = pq.ParquetWriter(
                        temp_path,
                        table.schema,
                        compression="zstd",
                        use_dictionary=True,
                    )

                writer.write_table(table)

                total_rows += len(rows)

                print(
                    f"\r  {total_rows:,} rows",
                    end="",
                    flush=True,
                )

        finally:
            if writer is not None:
                writer.close()

        # Only mark the hour complete once the entire query succeeded.
        temp_path.replace(output_path)

        print(f"\nSaved {total_rows:,} rows -> {output_path}")

    except Exception:
        temp_path.unlink(missing_ok=True)
        raise

    finally:
        cursor.close()

Open the following URL in browser for the external authentication:
https://trino.opensky-network.org/oauth2/token/initiate/26ec64ed8e3222531ba7d4bdcfd21c286d05d859baa616dffbb3614ab403a2b9
  900,000 rows

KeyboardInterrupt: 